# Lesson 07 - പ്ലാനിംഗ് ഡിസൈൻ പാറ്റേൺ

ഈ നോട്ട്‌ബുക്ക് Microsoft Agent Framework ഉപയോഗിച്ച് AI ഏജന്റുകൾക്കുള്ള **പ്ലാനിംഗ് ഡിസൈൻ പാറ്റേൺ** വ്യക്തമാക്കുന്നു.
കഠിനമായ യാത്രാ അഭ്യർത്ഥനകളെ ഘടിത സബ്ടാസ്കുകളായി വിഭജിച്ച്, അവ വിദഗ്ധ ഏജന്റുകളെ നിയോഗിച്ച്,
ആയി രൂപപ്പെട്ട പദ്ധതിയെ നടപ്പിലാക്കുന്നത് നിങ്ങൾ പഠിക്കും — എല്ലാം Pydantic മോഡലുകൾ ഉപയോഗിക്കുന്ന ഘടിത ഔട്ട്പുട്ട് പ്രയോജനപ്പെടുത്തി.


## സെറ്റപ്പ്


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ജോലി വിഭജനം

ജോലി വിഭജനം പ്ലാനിംഗ് ഡിസൈൻ പാറ്റേണിന്റെ മുഖ്യഭാഗമാണ്. ഒരു കോംപ്ലക്സ് അഭ്യർത്ഥനയെ ഒറ്റ ഏജൻറ് പൂര്‍ണ്ണമായി കൈകാര്യം ചെയ്യാൻ പറയുകയിടത്തോളം,
പ്രശ്നം ചെറുതും നന്നായി പരിധിയിടപ്പെട്ട **ഉപജോലികൾ** ആയുളളവത്തില്‍ വിഭജിക്കുന്നു.
ഓരോ ഉപജോലിയും ഒരു വിദഗ്ധ ഏജൻറുകൾക്ക് (ഉദാഹരണത്തിന്, വിമാനങ്ങൾ, ഹോട്ടലുകൾ, പ്രവർത്തനങ്ങൾ) നിയോഗിക്കപ്പെടുന്നു, വ്യക്തമായ
പ്രാധാന്യവും ആശ്രിതക്രമവും നിഷ്‌ചയിച്ചിരിക്കുന്നു.

ഈ സമീപനം പല ഗുണങ്ങളും നൽകുന്നു:
- **വ്യക്തത**: ഓരോ ഉപജോളിക്കും ഒരു ഏകദേശം ഉത്തരവാദിത്തമുണ്ട്.
- **പറലലിസം**: സ്വതന്ത്ര ഉപജോലികൾ സമാന്തരമായി നടപ്പിലാക്കാം.
- **വിശ്വസനീയത**: തെറ്റുകൾ ഓരോ ഉപജോളികളിലേക്കു മാത്രം സിമിതമാകും.
- **ബജറ്റ് ട്രാക്കിംഗ്**: ചെലവുകൾ ഓരോ ഉപജോളിക്കും നിശ്ചയിച്ച് അനുമാനിച്ച് കത്ത് ചെയ്തു കൂട്ടിച്ചേർക്കുന്നു.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## സ്ട്രക്ചേഡ് ഔട്ട്പുട്ടിനുള്ള ഒരു പ്ലാനിംഗ് ഏജന്റ് സൃഷ്ടിക്കല്‍

പ്ലാനിംഗ് ഏജന്റ് ഒരു **ഫ്രണ്ട് ഡെസ്ക് കോഓർഡിനേറ്ററായി** പ്രവർത്തിക്കുന്നു. ഒരു ഉയർന്ന നിലവാരത്തിലുള്ള യാത്രാ അഭ്യർത്ഥന നൽകിയാൽ അത് ഒരു സ്ട്രക്ചേഡ് `TravelPlan` ഉൽപ്പാദിപ്പിക്കുന്നു — അഭ്യർത്ഥനയെ ഉപരാമാർത്ഥ്യങ്ങളായി തന്ത്രമായി വിഭജിച്ച്, മുൻഗണനകൾ നിശ്ചയിച്ച്, ആശ്രിതത്വങ്ങൾ തിരിച്ചറിഞ്ഞ്, ഒരു കോൺസിയർജ് അല്ലെങ്കിൽ നിർവഹണ ലെയർ ജോലി നിർവഹിക്കാൻ കഴിയും.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## വിദഗ്ധ ഉപകരണങ്ങളുമായി ഒരു યોજના നടപ്പാക്കൽ

ഫ്രണ്ട് ഡെസ്ക് ഏജൻറ് ഒരു ഘടിതമായ പദ്ധതി തയ്യാറാക്കി കഴിഞ്ഞാൽ, **കൺസിയർജ്ജ് ഏജൻറ്** അതിനെ നടപ്പിലാക്കുന്നു.  
ഓരോ വിദഗ്ധ ഉപകരണവും ഒരു ഉപകാര്യ വിഭാഗം (ഫ്ലൈറ്റുകൾ, ഹോട്ടലുകൾ, പ്രവർത്തനങ്ങൾ) കൈകാര്യം ചെയ്യുന്നു. കൺസിയർജ്ജ് പദ്ധതിയിലെ ഉപകാര്യങ്ങളെ ആശ്രിത ക്രമത്തിൽ مرورിച്ച്, ഓരോന്നും അനുയോജ്യമായ ഉപകരണത്തിലേക്ക് അയയ്ക്കുന്നു.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## സംക്ഷേപം

ഈ പാഠത്തിൽ നിങ്ങൾ AI ഏജന്റുകൾക്കായുള്ള **പ്ലാനിങ് ഡിസൈൻ പാറ്റേൺ** മനസിലാക്കി:

1. **ടാസ്ക് ഡികംപോസിഷൻ** — ഒരു ഫ്രണ്ട് ഡെസ്ക് പ്ലാനിങ് ഏജന്റ് ഒരു ജടിലമായ യാത്രാ അഭ്യർത്ഥന പൈഡാന്റിക് മോഡലുകൾ ഉപയോഗിച്ച് ഘടനാസമ്ബന്നമായ ഉപകാര്യങ്ങളായി വിഭജിച്ച്, പ്രാധാന്യങ്ങളും ആശ്രിതത്വങ്ങളും കൂടി ഓരോ വിദഗ്ധ ഏജന്റിനും നല്കുന്നു.
2. **ഘടനാപരമായ ഔട്ട്‌പുട്ട്** — `response_format` പാസ്സ് ചെയ്‌തുകൊണ്ട് ഏജന്റ് സ്വതന്ത്ര ഫോം ടെക്സ്റ്റിന് പകരം പരിശോധിക്കപ്പെട്ട ഒരു `TravelPlan` ഒബ്ജക്റ്റ് റിട്ടേൺ ചെയ്യുന്നു, ഇത് പിന്നീട് പ്രോസസിങ് വിശ്വസനീയമാക്കുന്നു.
3. **പ്ലാൻ എക്സിക്യൂഷൻ** — ഒരു കോൺസിയർജ് ഏജന്റ് ഉപകാര്യങ്ങളിലൂടെ വിദഗ്ധ ഉപകരണങ്ങൾ (`book_flight`, `reserve_hotel`, `book_activity`) ഉപയോഗിച്ച് പദ്ധതിയെ നടപ്പിലാക്കി ഫലം റിപ്പോർട്ട് ചെയ്യുന്നു.

ഈ പാറ്റേൺ *എന്ത് ചെയ്യണം* (പ്ലാനിംഗ്) എന്നതും *എങ്ങനെ ചെയ്യണം* (എക്സിക്യൂഷൻ) എന്നതും വേർതിരിക്കുമ്പോൾ, ഏജന്റുകൾ കൂടുതൽ മഡ്യൂളർ, ടെസ്റ്റബിൾ, വിപുലീകരിക്കാൻ എളുപ്പമാകുന്നതാണ്.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അസംബന്ധിത ഓർമ്മപ്പെടുത്തൽ**:  
ഈ രേഖ AI വിവർത്തന സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. നമുക്ക് ശരിതത്വം ഉറപ്പാക്കാൻ പരിശ്രമിക്കുമ്പോഴും, ഓട്ടോമാറ്റഡ് വിവർത്തനങ്ങളിൽ പിശകുകൾ അല്ലെങ്കിൽ അപൂർണക്കങ്ങൾ ഉണ്ടായേക്കാം എന്ന് ദയവായി ശ്രദ്ധിക്കുക. സാങ്കേതിക ഭാഷയിൽ ഉള്ള പ്രമാണം ആത്മവിശ്വാസമുള്ള ഉറവിടമായി കരുതപ്പെടണം. പ്രധാന വിവരങ്ങൾക്ക് വിദഗ്ദ്ധ മാനവ വിവർത്തനം നടത്തുന്നത് ശിപാർശ ചെയ്യപ്പെടുന്നു. ഈ വിവർത്തനം ഉപയോഗിക്കുന്നതിനാൽ ഉണ്ടാകാവുന്ന തെറ്റിദ്ധാരണകൾക്കോ മിഴിവ് മിണ്ടുകളിലേക്കോ വേണ്ടി ഞങ്ങൾ ഉത്തരവാദിത്വം വഹിക്കുന്നില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
